**Mount Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


**Clone and Enter DeiT Repository**

In [ ]:
!git clone https://github.com/facebookresearch/deit.git
%cd deit

**Install Required Python Packages**

In [ ]:
!pip install timm
!pip install submitit
!pip install opencv-python


**Install PyTorch and Torchvision**

In [ ]:
!pip install torch torchvision


**Define Consistency Loss Function (MSE)**

In [ ]:
# loss.py
with open("/content/deit/loss.py", "w") as f:
    f.write('''import torch.nn as nn

def consistency_loss_mse(view1, view2):
    """MSE loss between two augmented views"""
    loss_fn = nn.MSELoss()
    return loss_fn(view1, view2)
''')


**Define Data Augmentations (RaAug and DFDC_Selim)**

In [ ]:
with open("/content/deit/augmentations.py", "w") as f:
    f.write('''from torchvision import transforms
from PIL import Image
import random
import numpy as np
from io import BytesIO

# Custom Augmentation Components

class RandomJPEGCompression:
    """Applies random JPEG compression to simulate quality loss."""
    def __init__(self, quality=(30, 100)):
        self.quality = quality

    def __call__(self, img):
        buffer = BytesIO()
        q = random.randint(*self.quality)
        img.save(buffer, format="JPEG", quality=q)
        buffer.seek(0)
        return Image.open(buffer)


class AddGaussianNoise:
    """Adds Gaussian noise to an image."""
    def __init__(self, mean=0., std=0.1):
        self.mean = mean
        self.std = std

    def __call__(self, img):
        img_array = np.array(img).astype(np.float32) / 255.0
        noise = np.random.normal(self.mean, self.std, img_array.shape)
        noisy_img = np.clip(img_array + noise, 0, 1) * 255
        return Image.fromarray(noisy_img.astype(np.uint8))


class RandomErasing:
    """Applies random erasing to simulate occlusion."""
    def __init__(self, p=0.5, scale=(0.02, 0.2), ratio=(0.5, 2.0)):
        self.p = p
        self.scale = scale
        self.ratio = ratio

    def __call__(self, img):
        if random.random() > self.p:
            return img

        img_array = np.array(img)
        h, w = img_array.shape[:2]
        area = h * w

        for _ in range(10):
            target_area = random.uniform(*self.scale) * area
            aspect_ratio = random.uniform(*self.ratio)

            erase_h = int(round(np.sqrt(target_area * aspect_ratio)))
            erase_w = int(round(np.sqrt(target_area / aspect_ratio)))

            if erase_h < h and erase_w < w:
                top = random.randint(0, h - erase_h)
                left = random.randint(0, w - erase_w)
                img_array[top:top+erase_h, left:left+erase_w] = 0
                break

        return Image.fromarray(img_array)



# Augmentation Pipelines

def get_transform(name):
    """Returns the specified augmentation pipeline."""

    if name == 'raaug':
        # RaAug (Random Augmentation) — geometric + erasing diversity
        def raaug_transform(img):
            choice = random.choice(['none', 'erase', 'randcrop'])

            if choice == 'none':
                img = transforms.Resize((224, 224))(img)
            elif choice == 'erase':
                img = transforms.Resize((224, 224))(img)
                img = RandomErasing(p=1.0, scale=(0.02, 0.2), ratio=(0.5, 2.0))(img)
            else:
                img = transforms.RandomResizedCrop(
                    224,
                    scale=(1/1.3, 1.0),  # roughly (0.77, 1.0)
                    ratio=(0.9, 1.1)
                )(img)

            return transforms.ToTensor()(img)

        return transforms.Lambda(raaug_transform)


    elif name == 'dfdcselim':
        # DFDC_Selim — compression, blur, noise, and geometric shifts
        return transforms.Compose([
            transforms.RandomApply([RandomJPEGCompression(quality=(30, 100))], p=0.5),
            transforms.RandomApply([AddGaussianNoise(mean=0, std=0.1)], p=0.3),
            transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))], p=0.3),
            transforms.RandomApply([transforms.RandomAffine(degrees=0, translate=(0.1, 0.1))], p=0.3),
            transforms.RandomApply([transforms.RandomResizedCrop(224, scale=(0.8, 1.0), ratio=(0.95, 1.05))], p=0.3),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
        ])

    else:
        raise ValueError(f"Unknown transform name: {name}")
''')


**Define Evaluation Metrics: AUC and TDR**

In [ ]:

import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve

def compute_auc(y_true, y_scores):
    """
    Computes ROC AUC score.

    Args:
        y_true (list or np.array): Ground truth binary labels (0 or 1).
        y_scores (list or np.array): Predicted probabilities or scores.

    Returns:
        float: AUC score.
    """
    try:
        return roc_auc_score(y_true, y_scores)
    except ValueError:
        return 0.0  # If AUC cannot be computed

def compute_tdr(y_true, y_scores, fpr_threshold=0.1):
    """
    Computes True Detection Rate (TDR = TPR) at a given FPR threshold.

    Args:
        y_true (list or np.array): Ground truth binary labels (0 or 1).
        y_scores (list or np.array): Predicted probabilities or scores.
        fpr_threshold (float): The maximum allowable false positive rate.

    Returns:
        float: TDR (True Positive Rate) at the given FPR threshold.
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    mask = fpr <= fpr_threshold
    if not np.any(mask):
        return 0.0
    return float(np.max(tpr[mask]))


**Write Metrics Module to File**

In [ ]:
with open("/content/deit/metrics.py", "w") as f:
    f.write("""
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve

def compute_auc(y_true, y_scores):
    try:
        return roc_auc_score(y_true, y_scores)
    except ValueError:
        return 0.0

def compute_tdr(y_true, y_scores, fpr_threshold=0.1):
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    mask = fpr <= fpr_threshold
    if not np.any(mask):
        return 0.0
    return float(np.max(tpr[mask]))
""")


**Define Custom Dataset Loader from CSV**

In [ ]:
with open("/content/deit/datasets.py", "w") as f:
    f.write("""import os
import glob
import random
import pandas as pd
from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset

class ViTCoreDatasetFromCSV(Dataset):
    def __init__(self, csv_path, root_dir, transform1, transform2):
        self.transform1 = transform1
        self.transform2 = transform2
        self.samples = []

        df = pd.read_csv(csv_path)

        for row in df.itertuples(index=False):
            folder_path = os.path.join(root_dir, row.path)
            label = int(row.label)
            png_files = glob.glob(os.path.join(folder_path, "*.png"))

            if not png_files:
                print(f"[WARN] No .png files in: {folder_path}")
                continue

            for img_path in png_files:
                self.samples.append((img_path, label))

        print(f"[INFO] Dataset initialized with {len(self.samples)} total frames.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]

        try:
            image = Image.open(img_path).convert("RGB")
        except (UnidentifiedImageError, OSError):
            print(f"[!] Skipping unreadable image: {img_path}")
            return self.__getitem__((idx + 1) % len(self))

        view1 = self.transform1(image)
        view2 = self.transform2(image)
        return (view1, view2), label
""")


**Change Directory to DeiT Folder**

In [ ]:
%cd /content/deit


**List Contents of DeiT Directory**

In [ ]:
!ls /content/deit


**ViT-CORE Training and Validation Script**

In [ ]:
%%writefile /content/deit/train_vitcore.py

# Imports and System Setup
import sys
sys.path.append("/content/deit")
import argparse
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset
from PIL import Image, UnidentifiedImageError
from torchvision import transforms
from timm.models import vit_small_patch16_224
from collections import Counter
from tqdm.auto import tqdm
import glob
import atexit
import random
import numpy as np
from augmentations import get_transform
import csv
from sklearn.metrics import roc_auc_score
from metrics import compute_tdr
from collections import defaultdict
from pathlib import Path
import pandas as pd
from datasets import ViTCoreDatasetFromCSV

# Custom Dataset Definition
class ViTCoreDatasetFromCSV(Dataset):
    def __init__(self, csv_path, root_dir, transform1, transform2):
        self.root_dir = root_dir
        self.transform1 = transform1
        self.transform2 = transform2
        self.samples = []

        df = pd.read_csv(csv_path)

        for i in range(len(df)):
            rel_img_path = df.iloc[i]['path']
            label = int(df.iloc[i]['label'])

            abs_img_path = os.path.join(self.root_dir, rel_img_path)

            if not os.path.exists(abs_img_path):
                print(f"[WARNING] Missing file: {abs_img_path}")
                continue
            self.samples.append((abs_img_path, label))


    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]

        try:
            image = Image.open(img_path).convert("RGB")
        except (UnidentifiedImageError, OSError):
            print(f"[!] Skipping unreadable image: {img_path}")
            return self.__getitem__((idx + 1) % len(self))

        view1 = self.transform1(image)

        if self.transform2:
            view2 = self.transform2(image)
            return (view1, view2), label
        else:
            return view1, label


# Argument Parsing
parser = argparse.ArgumentParser()
parser.add_argument('--data-path', type=str, required=True)
parser.add_argument('--output_dir', type=str, required=True)
parser.add_argument('--batch-size', type=int, default=32)
parser.add_argument('--epochs', type=int, default=50)
parser.add_argument('--lr', type=float, default=1e-4)
parser.add_argument('--lambda_consistency', type=float, default=2.0)
parser.add_argument('--consistency_loss', type=str, default='mse')
parser.add_argument('--use-raaug', action='store_true')
parser.add_argument('--use-dfdcselim', action='store_true')
parser.add_argument('--seed', type=int, default=42)
args = parser.parse_args()

# Reproducibility & Directories
os.makedirs(args.output_dir, exist_ok=True)
torch.manual_seed(args.seed)
random.seed(args.seed)
np.random.seed(args.seed)
gen = torch.Generator()
gen.manual_seed(args.seed)

if not (args.use_raaug and args.use_dfdcselim):
    raise ValueError("You must specify both --use-raaug and --use-dfdcselim")

# Transforms
transform_raaug = get_transform('raaug')
transform_dfdcselim = get_transform('dfdcselim')

# Training Dataset & Sampler
csv_train_path = "/content/drive/MyDrive/ViT-CORE-Datasets/ffpp/train_filtered.csv"


train_dataset = ViTCoreDatasetFromCSV(
    csv_path="/content/drive/MyDrive/ViT-CORE-Datasets/ffpp/train_filtered.csv",
    root_dir="/content/drive/MyDrive/ViT-CORE-Datasets/ffpp/train",
    transform1=transform_raaug,
    transform2=transform_dfdcselim
)

labels = [label for _, label in train_dataset.samples]
class_counts = Counter(labels)
sample_weights = torch.DoubleTensor([1.0 / class_counts[label] for label in labels])

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
    generator=torch.Generator().manual_seed(42)
)

dataloader = DataLoader(
    train_dataset,
    batch_size=args.batch_size,
    sampler=sampler,
    num_workers=0,
    pin_memory=False,
    drop_last=True
)

# Validation DataLoader
csv_val_path = "/content/drive/MyDrive/ViT-CORE-Datasets/ffpp/val_filtered.csv"
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

val_dataset = ViTCoreDatasetFromCSV(
    csv_path=csv_val_path,
    root_dir="/content/drive/MyDrive/ViT-CORE-Datasets/ffpp/val",
    transform1=val_transform,
    transform2=None
)

val_loader = DataLoader(
    val_dataset,
    batch_size=args.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

# Model, Optimizer, Losses
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = vit_small_patch16_224(pretrained=True)
model.head = nn.Linear(model.head.in_features, 2)
optimizer = optim.Adam(model.parameters(), lr=args.lr)
ce_loss = nn.CrossEntropyLoss()

if args.consistency_loss == 'mse':
    from loss import consistency_loss_mse as consistency_loss_fn
elif args.consistency_loss == 'cosine':
    from loss import consistency_loss_cosine as consistency_loss_fn
else:
    raise ValueError(f"Unsupported consistency loss: {args.consistency_loss}")

# Checkpoint & Resume Logic
start_epoch = 0
resume_batch_idx = 0
total_loss = total_ce = total_cons = correct = total = 0

checkpoint_path = os.path.join(args.output_dir, "vitcore_latest.pth")
best_model_path = os.path.join(args.output_dir, "vitcore_best.pth")
last_batch_path = os.path.join(args.output_dir, "last_batch.txt")
last_epoch_txt = os.path.join(args.output_dir, "last_epoch.txt")
metrics_path = os.path.join(args.output_dir, "accumulated_metrics.pth")
csv_path = os.path.join(args.output_dir, "vitcore_losses.csv")

if os.path.exists(checkpoint_path):

    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint.get('epoch', 0)
    best_auc = checkpoint.get('best_auc', 0.0)

if 'best_auc' not in locals():
    best_auc = 0.0

if os.path.exists(last_batch_path):
    with open(last_batch_path, "r") as f:
        resume_batch_idx = int(f.read())

if os.path.exists(metrics_path):
    print(f"[Resume] Loading metrics from {metrics_path}")
    metrics_checkpoint = torch.load(metrics_path, weights_only=False)
    total_loss = metrics_checkpoint.get('total_loss', 0.0)
    total_ce = metrics_checkpoint.get('total_ce', 0.0)
    total_cons = metrics_checkpoint.get('total_cons', 0.0)
    correct = metrics_checkpoint.get('correct', 0)
    total = metrics_checkpoint.get('total', 0)

model.to(device)

# Exit and Save Model
def save_on_exit():
    torch.save({
        'epoch': start_epoch,  # fallback if epoch is unknown
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': total_loss,
        'total_ce': total_ce,
        'total_cons': total_cons,
        'correct': correct,
        'total': total,
        'best_auc': best_auc
    }, os.path.join(args.output_dir, "vitcore_exit.pth"))
    print("[Exit Save] Model saved on exit.")


atexit.register(save_on_exit)

# Initialize CSV if Missing
if not os.path.exists(csv_path):
    with open(csv_path, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "total_loss", "total_ce", "total_cons", "accuracy", "auc", "tdr@0.1", "tdr@0.01"])

# Training Loop Begins
SAVE_EVERY = 5
model.train()


# Setup best_auc tracking
best_auc = 0.0
resume_checkpoint = os.path.join(args.output_dir, "vitcore_best.pth")
if os.path.exists(resume_checkpoint):
    checkpoint = torch.load(resume_checkpoint, map_location="cpu", weights_only=False)
    if "best_auc" in checkpoint:
        best_auc = checkpoint["best_auc"]

print("Starting training loop...")

try:
    for epoch in range(start_epoch, args.epochs):
        lambda_cons = args.lambda_consistency
        print(f"Epoch {epoch+1}/{args.epochs}, Lambda Consistency: {lambda_cons:.4f}")

        if resume_batch_idx == 0:
            total_loss = total_ce = total_cons = correct = total = 0

        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}")
        all_labels = []
        all_probs = []

        # Batch-wise Training Section
        for batch_idx, ((view1, view2), labels) in enumerate(pbar):
            if batch_idx < resume_batch_idx:
                continue

            view1, view2, labels = view1.to(device), view2.to(device), labels.to(device)

            optimizer.zero_grad()
            preds1 = model(view1)
            preds2 = model(view2)

            loss_ce = ce_loss(preds1, labels)
            loss_cons = consistency_loss_fn(preds1, preds2)
            loss = loss_ce + lambda_cons * loss_cons

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_ce += loss_ce.item()
            total_cons += loss_cons.item()

            _, predicted = torch.max(preds1, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

            probs = torch.softmax(preds1, dim=1)[:, 1].detach().cpu().numpy()
            all_probs.extend(probs.tolist())
            all_labels.extend(labels.detach().cpu().numpy().tolist())

            avg_loss = total_loss / (total / args.batch_size)
            accuracy = 100 * correct / total if total else 0.0
            pbar.set_postfix({"loss": f"{avg_loss:.3f}", "acc": f"{accuracy:.2f}%"})
            sys.stdout.flush()

            # Save intermediate checkpoint
            if batch_idx % 100 == 0 and batch_idx != 0:

                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'loss': total_loss,
                    'total_ce': total_ce,
                    'total_cons': total_cons,
                    'correct': correct,
                    'total': total,
                }, checkpoint_path)
                with open(last_batch_path, "w") as f:
                    f.write(str(batch_idx))
                torch.save({
                    'total_loss': total_loss,
                    'total_ce': total_ce,
                    'total_cons': total_cons,
                    'correct': correct,
                    'total': total,
                    'best_auc': best_auc
                }, metrics_path)
                print(f"[Checkpoint] Saved at batch {batch_idx} (Epoch {epoch+1})")

                accuracy = 100 * correct / total
        print(f"Epoch {epoch+1} Complete: Loss {total_loss:.4f}, Accuracy {accuracy:.2f}%")


        # Validation Phase
        model.eval()
        val_total = 0
        val_correct = 0
        val_labels = []
        val_probs = []

        with torch.no_grad():
            for val_imgs, val_labels_batch in tqdm(val_loader, desc=f"[VAL] Epoch {epoch+1}"):
                if isinstance(val_imgs, tuple):
                    val_imgs = val_imgs[0]
                val_imgs = val_imgs.to(device)
                val_labels_batch = val_labels_batch.to(device)

                val_preds = model(val_imgs)
                _, val_predicted = torch.max(val_preds, 1)
                val_correct += (val_predicted == val_labels_batch).sum().item()
                val_total += val_labels_batch.size(0)

                val_prob_batch = torch.softmax(val_preds, dim=1)[:, 1].detach().cpu().numpy()
                val_probs.extend(val_prob_batch.tolist())
                val_labels.extend(val_labels_batch.detach().cpu().numpy().tolist())

        val_acc = 100 * val_correct / val_total
        val_auc = roc_auc_score(val_labels, val_probs)
        val_tdr_01 = compute_tdr(val_labels, val_probs, fpr_threshold=0.1)
        val_tdr_001 = compute_tdr(val_labels, val_probs, fpr_threshold=0.01)

        print(f"[VAL] Accuracy: {val_acc:.2f}%, AUC: {val_auc:.4f}, TDR@0.1: {val_tdr_01:.4f}, TDR@0.01: {val_tdr_001:.4f}")
        model.train()

        # Log Validation Metrics to csv
        val_log_path = os.path.join(args.output_dir, "val_metrics.csv")
        if epoch == 0 and not os.path.exists(val_log_path):
            with open(val_log_path, mode='w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(["Epoch", "Val_Accuracy", "Val_AUC", "TDR@0.1", "TDR@0.01"])

        with open(val_log_path, mode='a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([epoch + 1, f"{val_acc:.2f}", f"{val_auc:.4f}", f"{val_tdr_01:.4f}", f"{val_tdr_001:.4f}"])

        # Save Best Model
        if val_auc > best_auc:
            best_auc = val_auc
            save_path = os.path.join(args.output_dir, "vitcore_best.pth")
            torch.save({
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "epoch": epoch + 1,
                "best_auc": best_auc
            }, save_path)


        # Save Latest Checkpoint
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': total_loss,
            'total_ce': total_ce,
            'total_cons': total_cons,
            'correct': correct,
            'total': total,
            'best_auc': best_auc
        }, checkpoint_path)

        torch.save({
            'total_loss': total_loss,
            'total_ce': total_ce,
            'total_cons': total_cons,
            'correct': correct,
            'total': total,
            'best_auc': best_auc
        }, metrics_path)


        # Log Epoch Metrics
        if total > 0:
            avg_loss = total_loss / (total / args.batch_size)
            with open(csv_path, mode='a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([
                    epoch + 1,
                    round(total_loss, 4),
                    round(total_ce, 4),
                    round(total_cons, 4),
                    round(accuracy, 2),
                    round(val_auc, 4),
                    round(val_tdr_01, 4),
                    round(val_tdr_001, 4)
                ])
        else:
            print(f"[Warning] Epoch {epoch+1}: total=0 — skipping CSV write.")


        # Reset for Next Epoch
        if os.path.exists(metrics_path):
            os.remove(metrics_path)

        with open(last_batch_path, "w") as f:
            f.write("0")
        resume_batch_idx = 0

        with open(last_epoch_txt, "w") as f:
            f.write(str(epoch + 1))

# Handle Crashes
except Exception as e:
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': total_loss,
        'total_ce': total_ce,
        'total_cons': total_cons,
        'correct': correct,
        'total': total,
        'best_auc': best_auc
    }, os.path.join(args.output_dir, "vitcore_crash.pth"))
    print(f"[CRASH] Training interrupted at epoch {epoch}. Model saved.")
    raise e


**Run ViT-CORE Script**

In [ ]:
!python /content/deit/train_vitcore.py \
  --data-path "/content/drive/MyDrive/ViT-CORE-Datasets/ffpp/train" \
  --output_dir "/content/drive/MyDrive/ViT-CORE-Experiments/ffpp_vitcore" \
  --batch-size 32 \
  --epochs 30 \
  --lr 1e-4 \
  --lambda_consistency 5 \
  --use-raaug \
  --use-dfdcselim \
  --seed 42


**FFpp Test Script (In-Domain)**

In [ ]:
# Imports
import sys
import os
import glob
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import roc_auc_score, accuracy_score
from PIL import Image, UnidentifiedImageError

# Paths
sys.path.append("/content/deit")
from augmentations import get_transform
from timm.models import vit_small_patch16_224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
csv_path = "/content/drive/MyDrive/ViT-CORE-Datasets/ffpp/test_filtered.csv"
root_dir = "/content/drive/MyDrive/ViT-CORE-Datasets/ffpp/test"
checkpoint_path = "/content/drive/MyDrive/ViT-CORE-Experiments/ffpp_vitcore/vitcore_best.pth"
batch_size = 32

# Dataset
class ViTCoreDatasetFromCSV_Test(Dataset):
    def __init__(self, csv_path, root_dir, transform1, transform2):
        self.transform1 = transform1
        self.transform2 = transform2
        self.samples = []

        df = pd.read_csv(csv_path)

        for row in df.itertuples(index=False):
            entry_path = os.path.join(root_dir, row.path)
            label = int(row.label)

            if os.path.isfile(entry_path) and entry_path.lower().endswith(".png"):
                self.samples.append((entry_path, label))
            else:
                png_files = glob.glob(os.path.join(entry_path, "*.png"))
                if not png_files:
                    print(f"[WARN] No .png files in: {entry_path}")
                    continue
                for img_path in png_files:
                    self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except (UnidentifiedImageError, OSError):
            print(f"[!] Skipping unreadable image: {img_path}")
            return self.__getitem__((idx + 1) % len(self))

        view1 = self.transform1(image)
        view2 = self.transform2(image)
        return (view1, view2), label

# Transforms
# View 1
test_transform_view1 = get_transform("dfdcselim")

# View 2: (no augmentation)
test_transform_view2 = transforms.Compose([
    transforms.Resize(256, interpolation=Image.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225)),
])

# Data Loader
dataset = ViTCoreDatasetFromCSV_Test(
    csv_path, root_dir,
    transform1=test_transform_view1,
    transform2=test_transform_view2
)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# Model
model = vit_small_patch16_224(pretrained=False, num_classes=2)
model.to(device)

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

if isinstance(checkpoint, dict):
    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif "model" in checkpoint and isinstance(checkpoint["model"], dict):
        state_dict = checkpoint["model"]
    else:
        raise RuntimeError("No valid model weights found in checkpoint.")
    model.load_state_dict(state_dict)
else:
    raise RuntimeError("Unexpected checkpoint format.")

model.eval()

# Testing
all_labels, all_preds, all_scores = [], [], []
with torch.no_grad():
    for (view1, _), labels in loader:  # only view1 is used for prediction
        view1, labels = view1.to(device), labels.to(device)

        outputs = model(view1)
        probs = torch.softmax(outputs, dim=1)[:, 1]

        preds = torch.argmax(outputs, dim=1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_scores.extend(probs.cpu().numpy())

# Metrics
if len(all_labels) > 0:
    auc = roc_auc_score(all_labels, all_scores)
    acc = accuracy_score(all_labels, all_preds)

    print(f"AUC: {auc:.4f}")
    print(f"Accuracy: {acc:.4f}")
else:
    print("[ERROR] No valid samples found for evaluation.")

**Celebdf Test Script (Cross-Domain)**

In [ ]:
# Imports
import sys
import os
import glob
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import roc_auc_score
from PIL import Image, UnidentifiedImageError

# Paths
sys.path.append("/content/deit")
from augmentations import get_transform
from timm.models import vit_small_patch16_224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
csv_path = "/content/drive/MyDrive/ViT-CORE-Datasets/celebdf/test_filtered.csv"
root_dir = "/content/drive/MyDrive/ViT-CORE-Datasets/celebdf/test"
checkpoint_path = "/content/drive/MyDrive/ViT-CORE-Experiments/ffpp_vitcore/vitcore_best.pth"
batch_size = 32

# Dataset
class ViTCoreDatasetFromCSV_Test(Dataset):
    def __init__(self, csv_path, root_dir, transform1, transform2):
        self.transform1 = transform1
        self.transform2 = transform2
        self.samples = []

        df = pd.read_csv(csv_path)

        for row in df.itertuples(index=False):
            entry_path = os.path.join(root_dir, row.path)
            label = int(row.label)

            if os.path.isfile(entry_path) and entry_path.lower().endswith(".png"):
                self.samples.append((entry_path, label))
            else:
                png_files = glob.glob(os.path.join(entry_path, "*.png"))
                if not png_files:
                    print(f"[WARN] No .png files in: {entry_path}")
                    continue
                for img_path in png_files:
                    self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except (UnidentifiedImageError, OSError):
            print(f"[!] Skipping unreadable image: {img_path}")
            return self.__getitem__((idx + 1) % len(self))

        view1 = self.transform1(image)
        view2 = self.transform2(image)
        return (view1, view2), label

# Transforms
# View 1
test_transform_view1 = get_transform("dfdcselim")

# View 2: (no augmentation)
test_transform_view2 = transforms.Compose([
    transforms.Resize(256, interpolation=Image.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225)),
])

# Data Loader
dataset = ViTCoreDatasetFromCSV_Test(
    csv_path, root_dir,
    transform1=test_transform_view1,
    transform2=test_transform_view2
)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# Model
model = vit_small_patch16_224(pretrained=False, num_classes=2)
model.to(device)

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

if isinstance(checkpoint, dict):
    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif "model" in checkpoint and isinstance(checkpoint["model"], dict):
        state_dict = checkpoint["model"]
    else:
        raise RuntimeError("No valid model weights found in checkpoint.")
    model.load_state_dict(state_dict)
else:
    raise RuntimeError("Unexpected checkpoint format.")

model.eval()

# Testing
all_labels, all_preds, all_scores = [], [], []
with torch.no_grad():
    for (view1, _), labels in loader:  # only view1 is used for prediction
        view1, labels = view1.to(device), labels.to(device)

        outputs = model(view1)
        probs = torch.softmax(outputs, dim=1)[:, 1]

        preds = torch.argmax(outputs, dim=1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_scores.extend(probs.cpu().numpy())

# Metrics
if len(all_labels) > 0:
    auc = roc_auc_score(all_labels, all_scores)

    print(f"AUC: {auc:.4f}")
else:
    print("[ERROR] No valid samples found for evaluation.")

**DFDC-Preview Test Script (Cross-Domain)**

In [ ]:
# Imports
import sys
import os
import glob
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import roc_auc_score
from PIL import Image, UnidentifiedImageError

# Paths
sys.path.append("/content/deit")
from augmentations import get_transform
from timm.models import vit_small_patch16_224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
csv_path = "/content/drive/MyDrive/ViT-CORE-Datasets/dfdc/test_filtered.csv"
root_dir = "/content/drive/MyDrive/ViT-CORE-Datasets/dfdc/test"
checkpoint_path = "/content/drive/MyDrive/ViT-CORE-Experiments/ffpp_vitcore/vitcore_best.pth"
batch_size = 32

# Dataset
class ViTCoreDatasetFromCSV_Test(Dataset):
    def __init__(self, csv_path, root_dir, transform1, transform2):
        self.transform1 = transform1
        self.transform2 = transform2
        self.samples = []

        df = pd.read_csv(csv_path)

        for row in df.itertuples(index=False):
            entry_path = os.path.join(root_dir, row.path)
            label = int(row.label)

            if os.path.isfile(entry_path) and entry_path.lower().endswith(".png"):
                self.samples.append((entry_path, label))
            else:
                png_files = glob.glob(os.path.join(entry_path, "*.png"))
                if not png_files:
                    print(f"[WARN] No .png files in: {entry_path}")
                    continue
                for img_path in png_files:
                    self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except (UnidentifiedImageError, OSError):
            print(f"[!] Skipping unreadable image: {img_path}")
            return self.__getitem__((idx + 1) % len(self))

        view1 = self.transform1(image)
        view2 = self.transform2(image)
        return (view1, view2), label

# Transforms
# View 1
test_transform_view1 = get_transform("dfdcselim")

# View 2: (no augmentation)
test_transform_view2 = transforms.Compose([
    transforms.Resize(256, interpolation=Image.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225)),
])

# Data Loader
dataset = ViTCoreDatasetFromCSV_Test(
    csv_path, root_dir,
    transform1=test_transform_view1,
    transform2=test_transform_view2
)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# Model
model = vit_small_patch16_224(pretrained=False, num_classes=2)
model.to(device)

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

if isinstance(checkpoint, dict):
    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif "model" in checkpoint and isinstance(checkpoint["model"], dict):
        state_dict = checkpoint["model"]
    else:
        raise RuntimeError("No valid model weights found in checkpoint.")
    model.load_state_dict(state_dict)
else:
    raise RuntimeError("Unexpected checkpoint format.")

model.eval()

# Testing
all_labels, all_preds, all_scores = [], [], []
with torch.no_grad():
    for (view1, _), labels in loader:  # only view1 is used for prediction
        view1, labels = view1.to(device), labels.to(device)

        outputs = model(view1)
        probs = torch.softmax(outputs, dim=1)[:, 1]

        preds = torch.argmax(outputs, dim=1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_scores.extend(probs.cpu().numpy())

if len(all_labels) > 0:
    auc = roc_auc_score(all_labels, all_scores)

    print(f"AUC: {auc:.4f}")
else:
    print("[ERROR] No valid samples found for evaluation.")

**WildDeepfake Test Script (Cross-Domain)**

In [ ]:
# Imports
import sys
import os
import glob
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import roc_auc_score
from PIL import Image, UnidentifiedImageError

# Paths
sys.path.append("/content/deit")
from augmentations import get_transform
from timm.models import vit_small_patch16_224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
csv_path = "/content/drive/MyDrive/ViT-CORE-Datasets/wilddeepfake/test_filtered.csv"
root_dir = "/content/drive/MyDrive/ViT-CORE-Datasets/wilddeepfake/test"
checkpoint_path = "/content/drive/MyDrive/ViT-CORE-Experiments/ffpp_vitcore/vitcore_best.pth"
batch_size = 32

# Dataset
class ViTCoreDatasetFromCSV_Test(Dataset):
    def __init__(self, csv_path, root_dir, transform1, transform2):
        self.transform1 = transform1
        self.transform2 = transform2
        self.samples = []

        df = pd.read_csv(csv_path)

        valid_exts = (".png", ".jpg", ".jpeg")

        for row in df.itertuples(index=False):
            entry_path = os.path.join(root_dir, row.path)
            label = int(row.label)

            # Case 1: direct file path in CSV
            if os.path.isfile(entry_path) and entry_path.lower().endswith(valid_exts):
                self.samples.append((entry_path, label))

            # Case 2: directory in CSV, collect all .png/.jpg/.jpeg
            else:
                img_files = []
                for ext in valid_exts:
                    img_files.extend(glob.glob(os.path.join(entry_path, f"*{ext}")))
                if not img_files:
                    print(f"[WARN] No valid image files in: {entry_path}")
                    continue
                for img_path in img_files:
                    self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except (UnidentifiedImageError, OSError):
            print(f"[!] Skipping unreadable image: {img_path}")
            return self.__getitem__((idx + 1) % len(self))

        view1 = self.transform1(image)
        view2 = self.transform2(image)
        return (view1, view2), label


# Transforms
# View 1
test_transform_view1 = get_transform("dfdcselim")

# View 2: (no augmentation)
test_transform_view2 = transforms.Compose([
    transforms.Resize(256, interpolation=Image.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225)),
])

# Data Loader
dataset = ViTCoreDatasetFromCSV_Test(
    csv_path, root_dir,
    transform1=test_transform_view1,
    transform2=test_transform_view2
)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# Model
model = vit_small_patch16_224(pretrained=False, num_classes=2)
model.to(device)

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

if isinstance(checkpoint, dict):
    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif "model" in checkpoint and isinstance(checkpoint["model"], dict):
        state_dict = checkpoint["model"]
    else:
        raise RuntimeError("No valid model weights found in checkpoint.")
    model.load_state_dict(state_dict)
else:
    raise RuntimeError("Unexpected checkpoint format.")

model.eval()

# Testing
all_labels, all_preds, all_scores = [], [], []
with torch.no_grad():
    for (view1, _), labels in loader:  # only view1 is used for prediction
        view1, labels = view1.to(device), labels.to(device)

        outputs = model(view1)
        probs = torch.softmax(outputs, dim=1)[:, 1]

        preds = torch.argmax(outputs, dim=1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_scores.extend(probs.cpu().numpy())

# Metrics
if len(all_labels) > 0:
    auc = roc_auc_score(all_labels, all_scores)

    print(f"AUC: {auc:.4f}")
else:
    print("[ERROR] No valid samples found for evaluation.")

**Visualize Dual-View Augmentations (RaAug vs DFDCselim)**

In [ ]:

import matplotlib.pyplot as plt
from PIL import Image
from augmentations import get_transform
import random
import os
import pandas as pd
import torchvision.transforms.functional as TF

# Load CSV and pick a random image
sample_csv = "/content/drive/MyDrive/ViT-CORE-Datasets/ffpp/test_filtered.csv"
sample_root = "/content/drive/MyDrive/ViT-CORE-Datasets/ffpp/test"

df = pd.read_csv(sample_csv)
print(f"Loaded {len(df)} entries from CSV")

sample_row = df.sample(n=1).iloc[0]
rel_path = sample_row['path']
abs_path = os.path.join(sample_root, rel_path)

original = Image.open(abs_path).convert("RGB")

transform_raaug = get_transform('raaug')
transform_dfdcselim = get_transform('dfdcselim')

raaug = transform_raaug(original)
dfdc = transform_dfdcselim(original)

# Convert to PIL for plotting
raaug_pil = TF.to_pil_image(raaug)
dfdc_pil = TF.to_pil_image(dfdc)

# Plot
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(original)
plt.title("Original Image")
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(raaug_pil)
plt.title("RaAug Transform")
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(dfdc_pil)
plt.title("DFDCselim Transform")
plt.axis('off')

plt.tight_layout()
plt.show()


**Augmentation Views**

In [ ]:

import sys
import os
import glob
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError

sys.path.append("/content/deit")
from augmentations import get_transform
from timm.models import vit_small_patch16_224
from metrics import 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
csv_path = "/content/drive/MyDrive/ViT-CORE-Datasets/wilddeepfake/test_filtered.csv"
root_dir = "/content/drive/MyDrive/ViT-CORE-Datasets/wilddeepfake/test"
checkpoint_path = "/content/drive/MyDrive/ViT-CORE-Experiments/ffpp_vitcore/vitcore_best.pth"
batch_size = 32

class ViTCoreDatasetFromCSV_Test(Dataset):
    def __init__(self, csv_path, root_dir, transform1, transform2):
        self.transform1 = transform1
        self.transform2 = transform2
        self.samples = []

        df = pd.read_csv(csv_path)
        valid_exts = (".png", ".jpg", ".jpeg")

        for row in df.itertuples(index=False):
            entry_path = os.path.join(root_dir, row.path)
            label = int(row.label)

            
            if os.path.isfile(entry_path) and entry_path.lower().endswith(valid_exts):
                self.samples.append((entry_path, label))
            else:
                
                img_files = []
                for ext in valid_exts:
                    img_files.extend(glob.glob(os.path.join(entry_path, f"*{ext}")))
                if not img_files:
                    print(f"[WARN] No valid image files in: {entry_path}")
                    continue
                for img_path in img_files:
                    self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except (UnidentifiedImageError, OSError):
            print(f"[!] Skipping unreadable image: {img_path}")
            return self.__getitem__((idx + 1) % len(self))

        view1 = self.transform1(image)
        return view1, label


test_transform_view1 = get_transform("dfdcselim")

test_transform_view2 = transforms.Compose([
    transforms.Resize(256, interpolation=Image.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225)),
])


dataset = ViTCoreDatasetFromCSV_Test(
    csv_path, root_dir,
    transform1=test_transform_view1,
    transform2=test_transform_view2
)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)


model = vit_small_patch16_224(pretrained=False, num_classes=2)
model.to(device)

checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

if isinstance(checkpoint, dict):
    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif "model" in checkpoint and isinstance(checkpoint["model"], dict):
        state_dict = checkpoint["model"]
    else:
        raise RuntimeError("No valid model weights found in checkpoint.")
    model.load_state_dict(state_dict)
else:
    raise RuntimeError("Unexpected checkpoint format.")

model.eval()

all_labels, all_scores = [], []
with torch.no_grad():
    for images, labels in loader:  # only view1 is used
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)[:, 1]

        all_labels.extend(labels.cpu().numpy())
        all_scores.extend(probs.cpu().numpy())

if len(all_labels) > 0:
    fpr, tpr, _ = roc_curve(all_labels, all_scores)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(7, 6))
    plt.plot(fpr, tpr, color="blue", lw=2, label="ROC Curve")
    plt.plot([0, 1], [0, 1], color="gray", lw=2, linestyle="--")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("")
    plt.legend(loc="lower right")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.show()
else:
    print("[ERROR] No valid samples found for evaluation.")

In [ ]:

import sys
import os
import glob
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import roc_auc_score, confusion_matrix
from PIL import Image, UnidentifiedImageError
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("/content/deit")
from timm.models import vit_small_patch16_224

dataset_name = "FaceForensics++ (In-Domain)" 
csv_path = "/content/drive/MyDrive/ViT-CORE-Datasets/ffpp/test_filtered.csv"
root_dir = "/content/drive/MyDrive/ViT-CORE-Datasets/ffpp/test"
checkpoint_path = "/content/drive/MyDrive/ViT-CORE-Experiments/ffpp_vitcore/vitcore_best.pth"
output_dir = '/content/drive/MyDrive/ViT-CORE-Experiments/charts/'
batch_size = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(output_dir, exist_ok=True)

class ViTCoreTestDataset(Dataset):
    def __init__(self, csv_path, root_dir, transform):
        self.transform = transform
        self.samples = []
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            img_path = os.path.join(root_dir, row['path'])
            label = int(row['label'])
            if os.path.exists(img_path):
                self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except (UnidentifiedImageError, OSError):
            print(f"Skipping unreadable image: {img_path}")
            return torch.zeros((3, 224, 224)), label

        tensor_image = self.transform(image)
        return tensor_image, label

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

dataset = ViTCoreTestDataset(csv_path, root_dir, transform=test_transform)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

model = vit_small_patch16_224(pretrained=False, num_classes=2)
model.to(device)

checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint.get('model', checkpoint.get('model_state_dict')))
model.eval()
print(f"Model loaded from {checkpoint_path}")

all_labels, all_preds = [], []
with torch.no_grad():
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

print("Evaluation complete.")

if len(all_labels) > 0:
    cm = confusion_matrix(all_labels, all_preds)
    class_names = ['Real', 'Fake']
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues',
                annot_kws={"size": 14})

    plt.title(f'Confusion Matrix for {dataset_name}', fontsize=16, pad=20)
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.xticks(fontsize=11)
    plt.yticks(fontsize=11)

    output_path = os.path.join(output_dir, 'confusion_matrix_ffpp_in_domain.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"Confusion matrix saved to: {output_path}")
else:
    print("No samples were evaluated, skipping confusion matrix generation.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

OUTPUT_DIR = "/content/drive/MyDrive/ViT-CORE-Experiments/ffpp_vitcore/"  # Your output directory
training_log_path = os.path.join(OUTPUT_DIR, "vitcore_losses.csv")

try:
    train_df = pd.read_csv(training_log_path)
    print(f"Successfully loaded training log with {len(train_df)} total epochs.")
except FileNotFoundError as e:
    print("Error: Could not find the training log file. Check the file path.")
    print(e)
    exit()


train_df = train_df[train_df['epoch'] <= 30]
print(f"Filtered to the first {len(train_df)} epochs.")


print("Generating plot...")
plt.style.use('seaborn-v0_8-whitegrid')

plt.figure(figsize=(8, 6))
plt.plot(train_df['epoch'], train_df['total_ce'], label='Training CE Loss', marker='o')
plt.title('Training Loss', fontsize=16)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Cross-Entropy Loss', fontsize=12)
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
import sys
import os
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("/content/deit")
from timm.models import vit_small_patch16_224

CSV_PATH = "/content/drive/MyDrive/ViT-CORE-Datasets/ffpp/test_filtered.csv"
ROOT_DIR = "/content/drive/MyDrive/ViT-CORE-Datasets/ffpp/test"
CHECKPOINT_PATH = "/content/drive/MyDrive/ViT-CORE-Experiments/ffpp_vitcore/vitcore_best.pth"

BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class FFppTestDataset(Dataset):
    def __init__(self, csv_path, root_dir, transform):
        self.transform = transform
        self.samples = []
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            abs_path = os.path.join(root_dir, row['path'])
            if os.path.exists(abs_path):
                self.samples.append((abs_path, int(row['label'])))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except (UnidentifiedImageError, OSError):
            print(f"Skipping unreadable image: {img_path}")
            return self.__getitem__((idx + 1) % len(self))

        return self.transform(image), torch.tensor(label, dtype=torch.long)

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

dataset = FFppTestDataset(CSV_PATH, ROOT_DIR, test_transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model = vit_small_patch16_224(pretrained=False, num_classes=2)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
if 'model' in checkpoint:
    model.load_state_dict(checkpoint['model'])
elif 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
else:
    model.load_state_dict(checkpoint)

model.to(DEVICE)
model.eval()

all_labels = []
all_scores = []

print(f"Running inference on {len(dataset)} test samples...")
with torch.no_grad():
    for images, labels in tqdm(loader, desc="Predicting"):
        images = images.to(DEVICE)

        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)[:, 1]

        all_scores.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_scores = np.array(all_scores)
all_labels = np.array(all_labels)

real_scores = all_scores[all_labels == 0]

fake_scores = all_scores[all_labels == 1]

print(f"\nFound {len(real_scores)} real samples and {len(fake_scores)} fake samples.")

print("Generating plot...")
plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(12, 7))

sns.histplot(fake_scores, color="coral", label='Fake Samples (True Label = 1)', kde=True, bins=50)

sns.histplot(real_scores, color="dodgerblue", label='Real Samples (True Label = 0)', kde=True, bins=50)

plt.title('Distribution of Prediction Confidence Scores (FF++)', fontsize=16)
plt.xlabel('Predicted Probability of Being FAKE', fontsize=12)
plt.ylabel('Number of Samples', fontsize=12)
plt.xlim(0, 1)
plt.legend()
plt.show()

In [ ]:
import sys
import os
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

sys.path.append("/content/deit")
from timm.models import vit_small_patch16_224

CHECKPOINT_PATH = "/content/drive/MyDrive/ViT-CORE-Experiments/ffpp_vitcore/vitcore_best.pth"
BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


DATASET_CONFIGS = {
    "Celeb-DF": {
        "csv_path": "/content/drive/MyDrive/ViT-CORE-Datasets/celebdf/test_filtered.csv",
        "root_dir": "/content/drive/MyDrive/ViT-CORE-Datasets/celebdf/test"
    },
    "DFDC-P": {
        "csv_path": "/content/drive/MyDrive/ViT-CORE-Datasets/dfdc/test_filtered.csv",
        "root_dir": "/content/drive/MyDrive/ViT-CORE-Datasets/dfdc/test"
    },
    "WildDeepfake": {
        "csv_path": "/content/drive/MyDrive/ViT-CORE-Datasets/wilddeepfake/test_filtered.csv",
        "root_dir": "/content/drive/MyDrive/ViT-CORE-Datasets/wilddeepfake/test"
    }
}

class CrossDomainTestDataset(Dataset):
    def __init__(self, csv_path, root_dir, transform):
        self.transform = transform
        self.samples = []
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            full_path = os.path.join(root_dir, row['path'])
            if os.path.isdir(full_path):
                img_files = [os.path.join(full_path, f) for f in os.listdir(full_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
                for img_path in img_files:
                       self.samples.append((img_path, int(row['label'])))
            elif os.path.exists(full_path):
                self.samples.append((full_path, int(row['label'])))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except (UnidentifiedImageError, OSError):
            print(f"Skipping unreadable image: {img_path}")
            return self.__getitem__((idx + 1) % len(self))

        return self.transform(image), torch.tensor(label, dtype=torch.long)


test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

model = vit_small_patch16_224(pretrained=False, num_classes=2)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
if 'model' in checkpoint:
    model.load_state_dict(checkpoint['model'])
else:
    model.load_state_dict(checkpoint.get('model_state_dict', checkpoint))
model.to(DEVICE)
model.eval()

def get_predictions(data_loader):
    all_labels, all_scores = [], []
    with torch.no_grad():
        for images, labels in tqdm(data_loader, desc="Predicting"):
            images = images.to(DEVICE)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            all_scores.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return np.array(all_labels), np.array(all_scores)

roc_results = {}
print("Processing datasets to generate ROC data...")

for name, config in DATASET_CONFIGS.items():
    print(f"\n--- Processing {name} ---")
    dataset = CrossDomainTestDataset(config["csv_path"], config["root_dir"], test_transform)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    labels, scores = get_predictions(loader)

    fpr, tpr, _ = roc_curve(labels, scores)
    roc_auc = auc(fpr, tpr)

    roc_results[name] = {"fpr": fpr, "tpr": tpr, "auc": roc_auc}
    print(f"Finished {name}, AUC: {roc_auc:.4f}")

print("\nGenerating combined ROC plot...")
plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(10, 8))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c'] # Blue, Orange, Green
for (name, result), color in zip(roc_results.items(), colors):
    # The 'label' argument has been added back to this line
    plt.plot(result['fpr'], result['tpr'], color=color, lw=2,
             label=f'{name}')


plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)

plt.legend(loc="lower right", fontsize=11) 
plt.grid(True)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

OUTPUT_DIR = "/content/drive/MyDrive/ViT-CORE-Experiments/ffpp_vitcore/"
training_log_path = os.path.join(OUTPUT_DIR, "vitcore_losses.csv")


try:
    df = pd.read_csv(training_log_path)
    print(f"Successfully loaded log file with {len(df)} total epochs.")

    
    df = df[df['epoch'] <= 30] 

    print(f"Filtered data to show the first {len(df)} epochs.")

except FileNotFoundError as e:
    print(f"Error: Could not find the log file. Make sure the path is correct.")
    print(e)
    exit()

print("Generating plot...")
plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(12, 7))

plt.plot(df['epoch'], df['total_loss'], label='Total Loss', color='blue', marker='o')
plt.plot(df['epoch'], df['total_ce'], label='Classification Loss (CE)', color='coral', marker='o')
plt.plot(df['epoch'], df['total_cons'], label='Consistency Loss', color='green', marker='o')

plt.title('Breakdown of Loss Components', fontsize=16)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.grid(True)
plt.show()